Підготовка

In [1]:
import os, re, glob, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

os.makedirs("results/figures", exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)

ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
spw = (y[m_tr] == 0).sum() / (y[m_tr] == 1).sum()

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p),
            "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k,
            "precision@1%": y_true[top1].mean(),
            "base_rate": y_true.mean()}

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")

def run_xgb_X(Xf, seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                      eval_metric="aucpr", early_stopping_rounds=50,
                      random_state=seed, **DEV)
    m.fit(Xf[m_tr], y[m_tr], eval_set=[(Xf[m_va], y[m_va])], verbose=False)
    r = metrics(y[m_te], m.predict_proba(Xf[m_te])[:, 1])
    r.update(seed=seed, best_iter=m.best_iteration, val_aucpr=m.best_score)
    return r, m

print("готово:", X.shape, "| xgboost", xgb.__version__)

готово: (590540, 434) | xgboost 3.4.1


Граф, таблиці сусідів, мітки

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda")
split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))

# ---- граф v1 (як у ноутбуці 05) ----
def build_edges(cols, K, W_days):
    valid = df[cols].notna().all(axis=1).values
    idx = np.flatnonzero(valid); sub = df.loc[valid, cols]
    S, D = [], []
    for pos in sub.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K, np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo; tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    return np.concatenate(S), np.concatenate(D)

REL = {"u3": (["card1", "addr1", "P_emaildomain"], 50, 30),
       "u2": (["card1", "addr1"], 50, 30),
       "c1": (["card1"], 20, 30)}
R_NAMES = list(REL); KMAX = {k: v[1] for k, v in REL.items()}
edges = {k: build_edges(*v) for k, v in REL.items()}
for k, (s, d) in edges.items():
    assert (rank[s] < rank[d]).all() and (s != d).all()
    assert (split_of[s[split_of[d] == 0]] == 0).all()
    print(k, f"ребер {len(s):,}")

# ---- таблиці сусідів на GPU: NBR[r] [N,K] (-1 = порожньо), DEG[r] [N] ----
NBR, DEG = [], []
for rn in R_NAMES:
    s, d = edges[rn]
    o = np.argsort(d, kind="stable"); ds, ss = d[o], s[o]
    cnt = np.bincount(d, minlength=N); start = np.cumsum(cnt) - cnt
    pos = np.arange(len(ds)) - start[ds]
    assert pos.max() < KMAX[rn]
    tab = np.full((N, KMAX[rn]), -1, dtype=np.int32); tab[ds, pos] = ss
    NBR.append(torch.from_numpy(tab).to(dev)); DEG.append(torch.from_numpy(cnt.astype(np.int64)).to(dev))

# ---- відкладені мітки сусідів, δ = 7 (як у ноутбуці 06) ----
KEYS = {"c1": ["card1"], "u2": ["card1", "addr1"], "u3": ["card1", "addr1", "P_emaildomain"]}
DL = 7
def label_feats(cols):
    valid = df[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    out = {k: np.full(N, np.nan, "float32") for k in ("nprior", "nk", "fk")}
    for pos in df.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        k = np.minimum(np.searchsorted(tt, tt - DL * 86400, side="right"), a)
        out["nprior"][rows] = a; out["nk"][rows] = k; out["fk"][rows] = cs[k]
    return out

LF = {}
for rel, cols in KEYS.items():
    for k, v in label_feats(cols).items(): LF[f"{rel}_{k}"] = v
LF = pd.DataFrame(LF)
for rel in KEYS:
    LF[f"{rel}_rate"] = (LF[f"{rel}_fk"] / LF[f"{rel}_nk"].where(LF[f"{rel}_nk"] > 0)).astype("float32")

# швидка перевірка «в лоб» на 60 рядках на відношення
rng = np.random.default_rng(1); bad = 0
for rel, cols in KEYS.items():
    cand = np.flatnonzero(df[cols].notna().all(axis=1).values)
    for i in rng.choice(cand, 60, replace=False):
        same = np.ones(N, bool)
        for c in cols: same &= (df[c].values == df[c].values[i])
        known = same & (ts <= ts[i] - DL * 86400)
        if (int(known.sum()) != int(LF[f"{rel}_nk"].values[i]) or
                int(y[known].sum()) != int(LF[f"{rel}_fk"].values[i])): bad += 1
assert bad == 0
print("граф і мітки готові, розбіжностей 0")

u3 ребер 5,295,877
u2 ребер 12,260,904
c1 ребер 9,739,081
граф і мітки готові, розбіжностей 0


Ознаки для нейромережі

In [3]:
from sklearn.preprocessing import QuantileTransformer
Xn = X.values.astype("float32")
grp = lambda p: [i for i, c in enumerate(X.columns) if re.fullmatch(p, c)]
nanf = np.stack([np.isnan(Xn[:, g]).mean(1) for g in
                 (grp(r"V\d+"), grp(r"D\d+"), grp(r"id_\d+"), grp(r"M\d+"))], 1).astype("float32")
qt = QuantileTransformer(n_quantiles=200, output_distribution="normal",
                         subsample=200_000, random_state=0).fit(Xn[m_tr])
Xt = np.hstack([np.nan_to_num(qt.transform(Xn), nan=0.0).clip(-5, 5).astype("float32"), nanf])

def lab_matrix():
    cols = []
    for r in KEYS:
        g = lambda k: np.nan_to_num(LF[f"{r}_{k}"].values)
        cols += [np.log1p(g("nk")), np.log1p(g("fk")), g("rate"),
                 (g("nk") > 0).astype("float32"), np.log1p(g("nprior"))]
    M = np.stack(cols, 1).astype("float32")
    return (M - M[m_tr].mean(0)) / (M[m_tr].std(0) + 1e-6)

XG_base = torch.from_numpy(Xt).half().to(dev)
XG_lab = torch.from_numpy(np.hstack([Xt, lab_matrix()])).half().to(dev)
print("A:", tuple(XG_base.shape), "| B:", tuple(XG_lab.shape))

A: (590540, 438) | B: (590540, 453)


Нова модель і тести на витік

In [4]:
ts_t = torch.as_tensor(ts, device=dev)                  # час, секунди
ST = {"y": torch.as_tensor(y, device=dev).float()}      # мітки (словник, щоб тест міг їх підміняти)
DLS = DL * 86400                                        # затримка міток, секунди (DL = 7)

def sample(nodes, r, s, gen=None):
    deg = DEG[r][nodes]
    u = torch.rand(nodes.shape[0], s, device=dev, generator=gen)
    idx = (u * deg.unsqueeze(1)).long().clamp_(max=KMAX[R_NAMES[r]] - 1)
    nb = NBR[r][nodes.unsqueeze(1), idx].long()
    valid = (deg > 0).unsqueeze(1).expand(-1, s) & (nb >= 0)
    return nb.clamp(min=0), valid

def hops(tgt, s1, s2, gen=None):
    B, R = tgt.shape[0], len(R_NAMES)
    n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
    N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
    n2, v2 = zip(*[sample(N1.reshape(-1), r, s2, gen) for r in range(R)])
    N2 = torch.stack(n2, 1).view(B, R, s1, R, s2)
    M2 = torch.stack(v2, 1).view(B, R, s1, R, s2) & M1[:, :, :, None, None]
    return N1, M1, N2, M2

def edge_feats(t_dst, nb, valid):
    """ознаки ребра: log(1+днів), мітка відома, відомий фрод, відомий чесний"""
    gap = t_dst - ts_t[nb]                               # секунди, ціле
    known = valid & (gap >= DLS)                         # те саме правило, що в бейзлайні
    yl = ST["y"][nb]
    f = torch.stack([torch.log1p(gap.clamp(min=0).float() / 86400.0),
                     known.float(), (known & (yl > 0.5)).float(), (known & (yl < 0.5)).float()], -1)
    return f * valid.unsqueeze(-1).float()

class RelSAGE2E(nn.Module):
    def __init__(self, F_in, mode="full", H=128, R=3, drop=0.2):
        super().__init__()
        self.R, self.ed = R, (1 if mode == "time" else 4)
        self.enc = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop))
        self.W1s, self.W2s = nn.Linear(H, H), nn.Linear(H, H)
        self.Wm1 = nn.ModuleList([nn.Linear(H + self.ed, H) for _ in range(R)])
        self.Wm2 = nn.ModuleList([nn.Linear(H + self.ed, H) for _ in range(R)])
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))

    @staticmethod
    def mmean(e, m, dim):
        m = m.unsqueeze(-1).to(e.dtype)
        return (e * m).sum(dim) / m.sum(dim).clamp(min=1)

    def forward(self, XG, tgt, s1, s2, gen=None):
        B, R = tgt.shape[0], self.R
        N1, M1, N2, M2 = hops(tgt, s1, s2, gen)
        f1 = edge_feats(ts_t[tgt][:, None, None], N1, M1)[..., :self.ed]
        f2 = edge_feats(ts_t[N1][:, :, :, None, None], N2, M2)[..., :self.ed]
        enc = lambda ids: self.enc(XG[ids].float())
        e_t = enc(tgt)
        e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
        e2 = enc(N2.reshape(-1)).view(B, R, s1, R, s2, -1)
        agg = 0                                           # шар 1 для сусідів 1-го кроку
        for r in range(R):
            m = F.relu(self.Wm1[r](torch.cat([e2[:, :, :, r], f2[:, :, :, r]], -1)))
            agg = agg + self.mmean(m, M2[:, :, :, r], 3)
        h1_u = F.relu(self.W1s(e1) + agg)
        agg = 0                                           # шар 1 для цілі
        for r in range(R):
            m = F.relu(self.Wm1[r](torch.cat([e1[:, r], f1[:, r]], -1)))
            agg = agg + self.mmean(m, M1[:, r], 1)
        h1_t = F.relu(self.W1s(e_t) + agg)
        agg = 0                                           # шар 2
        for r in range(R):
            m = F.relu(self.Wm2[r](torch.cat([h1_u[:, r], f1[:, r]], -1)))
            agg = agg + self.mmean(m, M1[:, r], 1)
        return self.out(F.relu(self.W2s(h1_t) + agg)).squeeze(-1)

# ---------- тести на витік ----------
sp_t = torch.as_tensor(split_of, device=dev)
te_nodes = np.flatnonzero(m_te)
T0 = ts[te_nodes[len(te_nodes) // 2]]
win = te_nodes[(ts[te_nodes] >= T0) & (ts[te_nodes] <= T0 + 86400)][:1500]   # ціль: вікно в 1 добу
Tcut = T0 + 86400 - DLS      # мітки вузлів із ts > Tcut не відомі жодній цілі вікна
tgt = torch.as_tensor(win, device=dev)
y0 = ST["y"].clone()

def logits(mdl, yy):
    ST["y"] = yy
    g = torch.Generator(device=dev); g.manual_seed(7)
    with torch.no_grad(): out = mdl(XG_base, tgt, 10, 5, g)
    ST["y"] = y0
    return out

for mode in ("time", "full"):
    torch.manual_seed(0)
    mdl = RelSAGE2E(XG_base.shape[1], mode).to(dev).eval()
    unk = y0.clone(); mu = ts_t > Tcut; unk[mu] = 1 - unk[mu]           # міняємо НЕвідомі мітки
    kn = y0.clone(); mk_ = ts_t <= Tcut; kn[mk_] = 1 - kn[mk_]          # міняємо відомі мітки
    a, b, c = logits(mdl, y0), logits(mdl, unk), logits(mdl, kn)
    assert torch.allclose(a, b, atol=1e-6), f"{mode}: невідомі мітки впливають на вихід (ВИТІК)"
    changed = not torch.allclose(a, c, atol=1e-6)
    assert changed == (mode == "full"), f"{mode}: відомі мітки {'не ' if not changed else ''}впливають"
    print(f"{mode}: невідомі мітки не впливають, відомі {'впливають' if changed else 'не впливають (так і має бути)'}")
print("тести на витік пройдено | параметрів (full):",
      sum(p.numel() for p in RelSAGE2E(XG_base.shape[1], 'full').parameters()))

time: невідомі мітки не впливають, відомі не впливають (так і має бути)
full: невідомі мітки не впливають, відомі впливають
тести на витік пройдено | параметрів (full): 191489


Навчання (спершу seed 0)

In [5]:
@torch.no_grad()
def predict_avg(model, XG, idx_np, s1, s2, seed, S=1, bs=2048):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev); acc = 0.0
    for s in range(S):
        gen = torch.Generator(device=dev); gen.manual_seed(1234 + 100 * seed + s)
        acc = acc + torch.cat([torch.sigmoid(model(XG, ids[i:i + bs], s1, s2, gen)).float().cpu()
                               for i in range(0, len(ids), bs)]).numpy()
    return acc / S

def run_nn(make_model, XG, seed, s1=10, s2=5, epochs=15, patience=4, bs=1024,
           lr=1e-3, final_S=1, tag=""):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(XG.shape[1]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    posw = torch.tensor(float(spw), device=dev)
    tr_idx = torch.as_tensor(np.flatnonzero(m_tr), device=dev)
    yt = torch.as_tensor(y, device=dev).float()
    va_idx, te_idx = np.flatnonzero(m_va), np.flatnonzero(m_te)
    best, best_ep, bad, state, t_all = -1.0, 0, 0, None, time.time()
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time()
        perm = tr_idx[torch.randperm(len(tr_idx), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(model(XG, b, s1, s2), yt[b], pos_weight=posw)
            opt.zero_grad(); loss.backward(); opt.step()
        v = average_precision_score(y[m_va], predict_avg(model, XG, va_idx, s1, s2, seed, S=1))
        print(f"{tag} seed {seed} епоха {ep}: val PR-AUC {v:.4f} | {time.time()-t0:.0f} с", flush=True)
        if v > best:
            best, best_ep, bad = v, ep, 0
            state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    r = metrics(y[m_te], predict_avg(model, XG, te_idx, s1, s2, seed, S=final_S))
    r.update(seed=seed, best_epoch=best_ep, val_aucpr=best, secs=time.time() - t_all)
    return r

mk = lambda mode: (lambda F_in: RelSAGE2E(F_in, mode))
res2 = []
for tag, mode in [("v2-time (час на ребрі)", "time"), ("v2 (мітки на ребрах)", "full")]:
    r = run_nn(mk(mode), XG_base, 0, tag=tag); r["variant"] = tag; res2.append(r)
    print(tag, {k: round(float(r[k]), 4) for k in ("pr_auc", "roc_auc", "recall@k", "precision@1%")}, flush=True)

v2 = pd.DataFrame(res2)[["variant", "pr_auc", "roc_auc", "recall@k", "precision@1%",
                         "best_epoch", "val_aucpr", "secs", "seed"]]
v2.to_csv("results/ieee_cis_relsage_v2_seed0.csv", index=False)
v2.round(4)

v2-time (час на ребрі) seed 0 епоха 1: val PR-AUC 0.4114 | 48 с
v2-time (час на ребрі) seed 0 епоха 2: val PR-AUC 0.4384 | 49 с
v2-time (час на ребрі) seed 0 епоха 3: val PR-AUC 0.4474 | 48 с
v2-time (час на ребрі) seed 0 епоха 4: val PR-AUC 0.4457 | 49 с
v2-time (час на ребрі) seed 0 епоха 5: val PR-AUC 0.4582 | 48 с
v2-time (час на ребрі) seed 0 епоха 6: val PR-AUC 0.4475 | 48 с
v2-time (час на ребрі) seed 0 епоха 7: val PR-AUC 0.4372 | 48 с
v2-time (час на ребрі) seed 0 епоха 8: val PR-AUC 0.4625 | 49 с
v2-time (час на ребрі) seed 0 епоха 9: val PR-AUC 0.4617 | 49 с
v2-time (час на ребрі) seed 0 епоха 10: val PR-AUC 0.4484 | 48 с
v2-time (час на ребрі) seed 0 епоха 11: val PR-AUC 0.4577 | 48 с
v2-time (час на ребрі) seed 0 епоха 12: val PR-AUC 0.4608 | 48 с
v2-time (час на ребрі) {'pr_auc': 0.436, 'roc_auc': 0.8661, 'recall@k': 0.4575, 'precision@1%': 0.7606}
v2 (мітки на ребрах) seed 0 епоха 1: val PR-AUC 0.4637 | 49 с
v2 (мітки на ребрах) seed 0 епоха 2: val PR-AUC 0.5129 | 48 с
v

,variant,pr_auc,roc_auc,recall@k,precision@1%,best_epoch,val_aucpr,secs,seed
0,v2-time (час на ребрі),0.4360,0.8661,0.4575,0.7606,8,0.4625,586.4486,0
1,v2 (мітки на ребрах),0.5142,0.9001,0.5213,0.8250,12,0.5496,728.2263,0
